# Audit Trail and Certification

This notebook demonstrates how to create audit trails and certification reports for systematic reviews using ASReview 5-Star.

## Topics Covered
1. PRISMA 2020 compliance
2. IRR certification
3. Stopping rule documentation
4. Complete audit report generation

In [ ]:
import asreview_5star as a5s
from datetime import datetime
import json

## 1. PRISMA 2020 Compliance Report

In [ ]:
# Example systematic review data
review_data = {
    "title": "Effectiveness of SGLT2 Inhibitors in Heart Failure: A Systematic Review",
    "registration": "PROSPERO CRD42023000001",
    "search_date": "2023-06-15",
    "databases": ["PubMed", "Embase", "Cochrane Library", "Web of Science"],
    "screeners": ["Reviewer A", "Reviewer B"],
    "adjudicator": "Senior Reviewer"
}

# PRISMA statistics
stats = a5s.prisma_stats(
    records_total=4250,
    records_duplicates=850,
    records_screened=3400,
    records_excluded_screening=3150,
    records_retrieved=250,
    records_not_retrieved=12,
    records_excluded_fulltext=213,
    records_from_databases=3800,
    records_from_registers=350,
    records_from_other=100,
    exclusion_reasons={
        "Wrong population": 65,
        "Wrong intervention": 52,
        "Wrong comparator": 38,
        "Wrong outcome": 28,
        "Wrong study design": 20,
        "Duplicate publication": 10
    }
)

flow_data = a5s.prisma_flow_data(stats)

print("PRISMA 2020 FLOW STATISTICS")
print("=" * 60)
print(f"\nIDENTIFICATION")
print(f"  Records from databases: {stats.records_identified_databases:,}")
print(f"  Records from registers: {stats.records_identified_registers:,}")
print(f"  Records from other sources: {stats.records_identified_other:,}")
print(f"  Duplicates removed: {stats.records_removed_duplicates:,}")
print(f"\nSCREENING")
print(f"  Records screened: {stats.records_screened:,}")
print(f"  Records excluded: {stats.records_excluded_screening:,}")
print(f"\nELIGIBILITY")
print(f"  Full-texts sought: {stats.reports_retrieved + stats.reports_not_retrieved:,}")
print(f"  Full-texts retrieved: {stats.reports_retrieved:,}")
print(f"  Full-texts not retrieved: {stats.reports_not_retrieved:,}")
print(f"  Reports excluded: {stats.reports_excluded:,}")
print(f"\nINCLUDED")
print(f"  Studies included: {stats.studies_included:,}")
print(f"\nSUMMARY METRICS")
print(f"  Yield rate: {flow_data['summary']['yield_rate']:.2f}%")
print(f"  Screening NNR: {flow_data['summary']['screening_nnr']:.1f}")

## 2. Inter-Rater Reliability Certification

In [ ]:
# Pilot IRR data (20% sample for calibration)
pilot_sample = {
    "sample_size": 200,
    "screener1": [1]*35 + [0]*165,  # 35 includes
    "screener2": [1]*32 + [0]*3 + [0]*162 + [1]*3  # 32 agrees, 3 false neg, 3 false pos
}

# Simulate some disagreement
import random
random.seed(42)

s1 = []
s2 = []
for i in range(200):
    if random.random() < 0.1:  # 10% relevant
        s1.append(1)
        s2.append(1 if random.random() < 0.90 else 0)  # 90% agreement on relevant
    else:
        s1.append(0)
        s2.append(0 if random.random() < 0.95 else 1)  # 95% agreement on irrelevant

# Calculate IRR
kappa_result = a5s.cohens_kappa(s1, s2)
agreement_result = a5s.percent_agreement(s1, s2)

print("INTER-RATER RELIABILITY CERTIFICATION")
print("=" * 60)
print(f"\nPilot Phase Results")
print(f"  Sample size: {len(s1)} records")
print(f"  Screener 1 includes: {sum(s1)}")
print(f"  Screener 2 includes: {sum(s2)}")
print(f"\nAgreement Metrics")
print(f"  Percent Agreement: {agreement_result.coefficient:.2%}")
print(f"  Cohen's Kappa: {kappa_result.coefficient:.4f}")
print(f"  95% CI: [{kappa_result.ci_lower:.4f}, {kappa_result.ci_upper:.4f}]")
print(f"  Interpretation: {kappa_result.interpretation}")
print(f"\nCertification Status: ", end="")

if kappa_result.coefficient >= 0.80:
    print("CERTIFIED - Excellent agreement")
    cert_status = "PASSED"
elif kappa_result.coefficient >= 0.60:
    print("CERTIFIED - Substantial agreement")
    cert_status = "PASSED"
elif kappa_result.coefficient >= 0.40:
    print("CONDITIONAL - Moderate agreement, recalibration recommended")
    cert_status = "CONDITIONAL"
else:
    print("FAILED - Insufficient agreement, recalibration required")
    cert_status = "FAILED"

## 3. Stopping Rule Documentation

In [ ]:
# Final screening state
screening_state = {
    "n_screened": 2800,
    "n_relevant": 250,
    "n_total": 3400,
    "consecutive_irrelevant": 127,
    "target_recall": 0.95
}

# Run all stopping rules
bayesian = a5s.bayesian_stopping(
    n_screened=screening_state["n_screened"],
    n_relevant=screening_state["n_relevant"],
    n_total=screening_state["n_total"],
    target_recall=screening_state["target_recall"]
)

sprt = a5s.sprt_stopping(
    n_screened=screening_state["n_screened"],
    n_relevant=screening_state["n_relevant"]
)

safe = a5s.safe_stopping(
    n_screened=screening_state["n_screened"],
    n_relevant=screening_state["n_relevant"],
    consecutive_irrelevant=screening_state["consecutive_irrelevant"],
    n_total=screening_state["n_total"],
    target_recall=screening_state["target_recall"]
)

consec = a5s.consecutive_irrelevant_stopping(
    consecutive_count=screening_state["consecutive_irrelevant"],
    threshold=100,
    n_relevant=screening_state["n_relevant"]
)

print("STOPPING RULE EVALUATION")
print("=" * 60)
print(f"\nScreening Progress")
print(f"  Screened: {screening_state['n_screened']:,} / {screening_state['n_total']:,} ({screening_state['n_screened']/screening_state['n_total']*100:.1f}%)")
print(f"  Relevant found: {screening_state['n_relevant']:,}")
print(f"  Consecutive irrelevant: {screening_state['consecutive_irrelevant']}")
print(f"\nStopping Rule Results")
print(f"  Bayesian (95% recall): {'STOP' if bayesian.should_stop else 'CONTINUE'} (confidence: {bayesian.confidence:.2%})")
print(f"  SPRT: {'STOP' if sprt.should_stop else 'CONTINUE'} ({sprt.details.get('decision', 'continue')})")
print(f"  SAFE: {'STOP' if safe.should_stop else 'CONTINUE'} (confidence: {safe.confidence:.2%})")
print(f"  Consecutive (100): {'STOP' if consec.should_stop else 'CONTINUE'} ({screening_state['consecutive_irrelevant']}/100)")

# Consensus decision
stop_votes = sum([bayesian.should_stop, sprt.should_stop, safe.should_stop, consec.should_stop])
print(f"\nConsensus: {stop_votes}/4 rules recommend stopping")
if stop_votes >= 3:
    print("Decision: STOP SCREENING")
    stop_decision = "STOP"
else:
    print("Decision: CONTINUE SCREENING")
    stop_decision = "CONTINUE"

## 4. Complete Audit Report

In [ ]:
# Generate complete audit report
audit_report = {
    "report_generated": datetime.now().isoformat(),
    "review": review_data,
    "prisma": {
        "identification": {
            "databases": stats.records_identified_databases,
            "registers": stats.records_identified_registers,
            "other": stats.records_identified_other,
            "duplicates_removed": stats.records_removed_duplicates
        },
        "screening": {
            "screened": stats.records_screened,
            "excluded": stats.records_excluded_screening
        },
        "eligibility": {
            "retrieved": stats.reports_retrieved,
            "not_retrieved": stats.reports_not_retrieved,
            "assessed": stats.reports_assessed,
            "excluded": stats.reports_excluded,
            "exclusion_reasons": stats.exclusion_reasons
        },
        "included": {
            "studies": stats.studies_included
        },
        "metrics": {
            "yield_rate": flow_data['summary']['yield_rate'],
            "screening_nnr": flow_data['summary']['screening_nnr']
        }
    },
    "irr_certification": {
        "pilot_sample_size": len(s1),
        "percent_agreement": agreement_result.coefficient,
        "cohens_kappa": kappa_result.coefficient,
        "kappa_ci": [kappa_result.ci_lower, kappa_result.ci_upper],
        "interpretation": kappa_result.interpretation,
        "certification_status": cert_status
    },
    "stopping_rules": {
        "screening_state": screening_state,
        "bayesian": {
            "should_stop": bayesian.should_stop,
            "confidence": bayesian.confidence,
            "details": bayesian.details
        },
        "sprt": {
            "should_stop": sprt.should_stop,
            "confidence": sprt.confidence,
            "decision": sprt.details.get('decision')
        },
        "safe": {
            "should_stop": safe.should_stop,
            "confidence": safe.confidence,
            "details": safe.details
        },
        "consecutive": {
            "should_stop": consec.should_stop,
            "count": screening_state['consecutive_irrelevant'],
            "threshold": 100
        },
        "consensus_decision": stop_decision,
        "rules_recommending_stop": stop_votes
    }
}

# Pretty print the report
print("COMPLETE AUDIT REPORT")
print("=" * 70)
print(json.dumps(audit_report, indent=2, default=str))

In [ ]:
# Save audit report
with open('audit_report.json', 'w') as f:
    json.dump(audit_report, f, indent=2, default=str)

print("Audit report saved to audit_report.json")

## Certification Summary

### Quality Assurance Checklist

| Criterion | Status | Notes |
|-----------|--------|-------|
| PROSPERO Registration | Registered | {review_data['registration']} |
| Dual Screening | Completed | Two independent screeners |
| IRR Certification | {cert_status} | Kappa = {kappa_result.coefficient:.3f} |
| Stopping Rule Consensus | {stop_decision} | {stop_votes}/4 rules agree |
| PRISMA Compliant | Yes | Flow diagram generated |

### Recommendations

1. **Archive this audit report** with your systematic review files
2. **Include PRISMA flow diagram** in your manuscript
3. **Report IRR statistics** in methods section
4. **Document stopping rationale** if screening was stopped early